In [ ]:
import os
import warnings
import copy
import numpy as np
import pandas as pd
import sqlite3
import healpy as hp
import matplotlib.pyplot as plt
import colorcet as cc
import skyproj
from IPython.display import display, HTML

from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
import astropy.units as u

from rubin_scheduler.scheduler import sim_runner
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.scheduler.schedulers import SimpleBandSched, DateSwapBandScheduler, CoreScheduler
from rubin_scheduler.scheduler.features import Conditions
from rubin_scheduler.scheduler.utils import SchemaConverter, run_info_table
from rubin_scheduler.site_models import Almanac
from rubin_scheduler.utils import ddf_locations, angular_separation, approx_ra_dec2_alt_az, Site
from rubin_scheduler.scheduler.utils import ObservationArray

import rubin_sim.maf as maf
from rubin_sim.data import get_baseline
import schedview.compute as schedview_compute

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.plot_utils as rn_plots
import rubin_nights.augment_visits as augment_visits
import rubin_nights.rubin_scheduler_addons as rn_sch
import rubin_nights.rubin_sim_addons as rn_sim
from rubin_nights.targets_and_visits import targets_and_visits

import importlib

from sv_survey import sv_support as svs
import lsst.ts.fbs.utils.maintel.sv_config as svc

#from sv_survey import dp2_support as dvs
#from sv_survey import dp2_config as dvc
#from sv_survey import fbs_config_dp2_survey 

from sv_survey import fbs_config_sv_survey 
#from sv_survey import fbs_config_sv_survey_generator_only


band_colors = rn_plots.PlotStyles.band_colors

#%load_ext memory_profiler

In [ ]:
endpoints = connections.get_clients("/Users/lynnej/.lsst/usdf_rsp")

refresh_visits = True

if refresh_visits:
    query = ( 
        "select *, q.* from cdb_lsstcam.visit1 left join cdb_lsstcam.visit1_quicklook as q on visit1.visit_id = q.visit_id "
        "where science_program = 'BLOCK-365' and observation_reason != 'block-t548' and observation_reason != 'field_survey_science'" 
          )
    visits = endpoints['consdb'].query(query)
    visits = augment_visits.augment_visits(visits, "lsstcam")
    
    #visits.to_hdf('vnow.h5', key='visits')
else:
    visits = pd.read_hdf('vnow.h5')

In [ ]:
make_plot = True
bad_visit_ids = augment_visits.fetch_excluded_visits("lsstcam")
gvisits = augment_visits.exclude_visits(visits, bad_visit_ids)
if make_plot:
    run_calc = True
    if run_calc:
        m_nvis = maf.CountMetric(col='obs_start_mjd', metric_name = "Nvisits")
        s = maf.HealpixSlicer(nside=64, lat_col='s_dec', lon_col='s_ra', rot_sky_pos_col_name = 'sky_rotation')
        constraint = ''
        opsvis = gvisits.to_records()
        nvis = maf.MetricBundle(m_nvis, s, constraint)
        g = maf.MetricBundleGroup({f'nvisits': nvis}, None)
        g.run_current(constraint, opsvis)
    
    fig = make_sv_plot(nvis, proj='McBryde', vmin=0, vmax=70, title=f"SV by {visits.day_obs.max()}")
    plt.savefig("sv_now.png", bbox_inches='tight')
    fig

In [ ]:
last_day_obs = 20250921
ln = visits.query("day_obs == @last_day_obs")
print(len(ln), ln.obs_start.min(), ln.obs_start.max())
print(ln.observation_reason.unique())
try:
    Time([ln.obs_start_mjd.min(), ln.obs_end_mjd.max()], format='mjd').iso
except (TypeError, ValueError):
    pass

In [ ]:
# vddfs = visits.query("observation_reason.str.contains('ddf')").copy()
# q = vddfs.groupby(["observation_reason", "day_obs"]).agg({'obs_start_mjd': 'mean', 's_ra': np.ptp, 's_dec': np.std}).reset_index("day_obs")
# for dname in q.index.unique():
#     qq = q.query("index == @dname")
#     plt.plot(qq.obs_start_mjd, qq.s_dec, label=dname)
# plt.legend()

In [ ]:
# ra = 2.26800  
# dec =-1.40465
# ra= 2.19135  
# dec= -1.62452 
# dist = angular_separation(visits.s_ra.values, visits.s_dec.values, ra, dec)
# tt = visits[['visit_id', 'day_obs', 'seq_num', 'band', 's_ra', 's_dec', 'sky_rotation', 
#              'zero_point_1s', 'sky_bg_median_mag', 'clouds', 'fwhm_eff']].copy()
# tt['dist'] = dist
# tt.query("dist < 2.5").sort_values(by='dist')

In [ ]:
## small field science query
# query = ( 
#     "select *, q.* from cdb_lsstcam.visit1 left join cdb_lsstcam.visit1_quicklook as q on visit1.visit_id = q.visit_id "
#     "where science_program = 'BLOCK-365' and observation_reason == 'field_survey_science'" 
#       )
# svisits = endpoints['consdb'].query(query)
# svisits = augment_visits.augment_visits(svisits, "lsstcam")
# svisits = augment_visits.exclude_visits(svisits, bad_visit_ids)
# print(len(svisits))
# sopsim = rn_sim.consdb_to_opsim(svisits)
# sopsim["note"] = sopsim["scheduler_note"].copy()
# ss = sopsim.groupby(["target_name", "band"]).agg({'seq_num': 'count'})
# ss.rename({"seq_num": "count"}, axis=1, inplace=True)
# ss = ss.reset_index('band').pivot(columns=["band"]).droplevel(0, axis=1)
# ss = ss[['u', 'g', 'r', 'i', 'z', 'y']]
# ss['all'] = ss.sum(axis=1)
# ss.query("all > 10").sort_values('all')

In [ ]:
# filename = "small_fields_20250818.db"
# con = sqlite3.connect(filename)
# sopsim.to_sql("observations", con, index=False, if_exists="replace")
# con.close()

In [ ]:
_ = importlib.reload(svs)
_ = importlib.reload(svc)
_ = importlib.reload(fbs_config_sv_survey)

In [ ]:
%%time
#%%memit
nside, starting_scheduler = fbs_config_sv_survey.get_scheduler()

In [ ]:
# ddf_requests = pd.DataFrame(starting_scheduler.survey_lists[0][0].obs_wanted)
# for ddf in ddf_requests.observation_reason.unique():
#     q = ddf_requests.query("observation_reason == @ddf")
#     plt.plot(q.mjd, q.band, '.')

In [ ]:
day_obs = 20250922
run_name = f"sv_{day_obs}"
out_dir = run_name

_ = importlib.reload(svs)

single_night = False

nside = 32
if single_night:
    survey_info = svs.survey_times(verbose=True, no_downtime=True, real_downtime=False, visits=None, day_obs=day_obs)
else:
    survey_info = svs.survey_times(verbose=True, no_downtime=False, real_downtime=True, visits=visits, day_obs=day_obs)

survey_info.update(svc.survey_footprint(survey_start_mjd=survey_info["survey_start"].mjd, nside=nside))

In [ ]:
# sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(20250828, sun_alt=-12)
# print(sunset.iso, sunrise.iso)
# match = np.where((survey_info['downtimes']['end'] > sunset.mjd) & (survey_info['downtimes']['start'] < sunrise.mjd))
# print(survey_info['downtimes'][match])
# vv = visits.query("day_obs == 20250828")
# vv.obs_start_mjd.min(), vv.obs_start_mjd.max()

In [ ]:
plt.plot(survey_info['dayobsmjd'], survey_info['avail_per_night']/survey_info['hours_in_night'])
plt.axvline(rn_dayobs.day_obs_to_time(day_obs).mjd, color='r', linestyle=':')
plt.xlabel("MJD", fontsize='large')
plt.ylabel("system availability", fontsize='large')

In [ ]:
downtime_ends = survey_info['downtimes']['end']
downtime_starts = survey_info['downtimes']['start']
_ = plt.hist((downtime_ends - downtime_starts) * 24)

In [ ]:
for start, end in survey_info['downtimes']:
    x = np.floor(start - 0.5)
    plt.plot((x, x), (start - x, end - x) , color='k') 
x = np.floor(survey_info['sunsets12'] - 0.5)
y = survey_info['sunsets12'] - x
plt.fill_between(x, 0.8, y)
x = np.floor(survey_info['sunrises12'] - 0.5)
y = survey_info['sunrises12'] - x
#plt.axvline(rn_dayobs.day_obs_to_time(day_obs).mjd, color='r', linestyle=':', linewidth=1.8)
#plt.axvline(np.floor(Time("2025-09-06T12:00:00").mjd - 0.5))
plt.fill_between(x, y, 1.6)
plt.ylim(0.9, 1.5)
plt.xlabel("MJD", fontsize='large')
plt.ylabel("Fraction of MJD", fontsize='large')
plt.savefig("onsky_downtime.png", bbox_inches='tight')

In [ ]:
sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt=-12)
print(sunset.iso, sunrise.iso)

In [ ]:
ddfs = ddf_locations(skycoords=True)
ddfs['edge1'] = SkyCoord(ra=300*u.deg, dec=-26*u.deg)
ddfs['edge2'] = SkyCoord(ra=324*u.deg, dec=-26*u.deg)
ddfs['edge3'] = SkyCoord(ra=300*u.deg, dec=-10*u.deg)
ddfs['edge4'] = SkyCoord(ra=324*u.deg, dec=-10*u.deg)
lsst_site = Site('LSST')
times = np.arange(survey_info['survey_start'].mjd, survey_info['survey_end'].mjd, 1/24)
times = np.arange(np.floor(Time.now().mjd - 0.5),survey_info['survey_end'].mjd, 0.25/24)
times = np.arange(sunset.mjd - 0.1, sunrise.mjd + 0.1, 0.05/24)
sunmoon = survey_info['almanac'].get_sun_moon_positions(times)
moon_ra = sunmoon['moon_RA']
moon_dec = sunmoon['moon_dec']
sun_alt = np.degrees(sunmoon['sun_alt'])
ddf_moon_dist = {}
ddf_alt = {}
for ddf in ddfs:
    ddf_moon_dist[ddf] = angular_separation(ddfs[ddf].ra.deg, ddfs[ddf].dec.deg, np.degrees(moon_ra), np.degrees(moon_dec))
    alt, az = approx_ra_dec2_alt_az(ddfs[ddf].ra.deg, ddfs[ddf].dec.deg, lsst_site.latitude, lsst_site.longitude, times, lmst=None)
    ddf_alt[ddf] = alt

In [ ]:
plt.figure()
for ddf in ['ELAISS1', 'ECDFS', 'edge1', 'edge2', 'edge3', 'edge4']:
    mask = np.where((sun_alt <= -12) & (ddf_alt[ddf] > 40))
    plt.plot(Time(times[mask], format='mjd').to_datetime(), ddf_moon_dist[ddf][mask], marker='.', linestyle='', label=ddf)
plt.legend(loc=(1.01, 0.5))
plt.axvline(sunset.to_datetime(), color='k', linestyle=':')
plt.axvline(sunrise.to_datetime(), color='k', linestyle=':')
plt.axhline(30)
plt.xticks(rotation=90)
plt.xlabel("MJD")
plt.ylabel("Distance to moon (deg)")

plt.figure()
for ddf in ['ELAISS1', 'ECDFS',  'edge1', 'edge2', 'edge3', 'edge4']:
    mask = np.where((sun_alt <= -12) & (ddf_alt[ddf] > 40))
    plt.plot(Time(times[mask], format='mjd').to_datetime(), ddf_alt[ddf][mask], marker='.', linestyle='', label=ddf)
plt.legend(loc=(1.01, 0.5))
plt.axvline(sunset.to_datetime(), color='k', linestyle=':')
plt.axvline(sunrise.to_datetime(), color='k', linestyle=':')
plt.axhline(30)
plt.xticks(rotation=90)
plt.xlabel("MJD")
plt.ylabel("Altitude (deg)")

In [ ]:
# What does the observatory look like?  (remember to set this back up if you change the downtimes)
setup_observatory = True
try:
    model_obs
except:
    setup_observatory = True 
    
if setup_observatory:
    # survey_info carries the downtime to the model_observatory
    if single_night:
        model_obs = svs.setup_observatory_summit(survey_info, clouds=False)
    else:
        model_obs = svs.setup_observatory_summit(survey_info, clouds=True)

reset = True
if reset:
    observatory = copy.deepcopy(model_obs)
    scheduler = copy.deepcopy(starting_scheduler)
# Filter scheduler - simply changes between ugriz and grizy depending on lunar phase
fs = DateSwapBandScheduler()

In [ ]:
if 'consdb_visits' not in survey_info:
    survey_info['consdb_visits'] = visits

In [ ]:
%%time
# Convert consdb visits to opsim visits and add to scheduler
initial_opsim = rn_sim.consdb_to_opsim(survey_info['consdb_visits'].query("day_obs < @day_obs"))
initial_opsim["note"] = initial_opsim["scheduler_note"].copy()
sch_obs = SchemaConverter().opsimdf2obs(initial_opsim)
scheduler.add_observations_array(sch_obs)

In [ ]:
#from rubin_nights import lfa_data
#uri = "https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2025/09/18/Scheduler:1_Scheduler:1_2025-09-19T05:58:43.595.p"
#sched, conditions = lfa_data.get_scheduler_snapshot(uri, at_usdf=True)

In [ ]:
%%time

rewards = False
scheduler.keep_rewards=rewards

# Start at dayobs sunset (minus a tiny bit) survey start?
#day_obs = survey_info['survey_start'].iso[0:10]
# or specific day - set earlier
#day_obs = rn_dayobs.day_obs_int_to_str(20250828)

sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt=-12)
start = sunset.mjd - 0.1 /24


if single_night:
    # end at sunrise
    end = sunrise.mjd #+ 3.1
else:
    # end at end of SV (probably)
    end = survey_info['survey_end'].mjd


with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    vals = sim_runner(
        observatory,
        scheduler,
        band_scheduler=fs,
        sim_start_mjd=start,
        sim_duration=end-start, 
        record_rewards=rewards,
        verbose=True,
    )
observatory = vals[0]
scheduler = vals[1]
observations = vals[2]
if len(vals) == 5:
    rewards = vals[3]
    obs_rewards = vals[4]

In [ ]:
if single_night:

    targets = pd.DataFrame(observations)
    
    from zoneinfo import ZoneInfo
    tz = ZoneInfo("Chile/Continental")
    tz_utc = ZoneInfo("UTC")
    telescope = "Simonyi"

    site = Site('LSST')
    almanac = Almanac()
    night_events = almanac.get_sunset_info(evening_date=rn_dayobs.day_obs_int_to_str(day_obs), longitude=site.longitude_rad)

    def mjd_to_datetime(mjd, scale='utc', timezone=tz):
        return Time(mjd, format='mjd', scale=scale).utc.to_datetime(timezone=timezone)
        
    eps = 1
    fig, ax = plt.subplots(figsize=(13, 8))
    ax_utc = ax.twiny()
    
    ax.set_title(f"{telescope} DAYOBS {day_obs}", pad=20)
    
    # Shade astronomical events
    ax.fill_between([mjd_to_datetime(night_events['sun_n12_setting']), 
                      mjd_to_datetime(night_events['sun_n18_setting'])],
                     2.5, 0.0, color='lightgray', alpha=0.3)
    ax.fill_between([mjd_to_datetime(night_events['sunset']), 
                     mjd_to_datetime(night_events['sun_n12_setting'])], 
                      2.5, 0.0, color='gray', alpha=0.3)
    ax.fill_between([mjd_to_datetime(night_events['sun_n18_rising']), 
                     mjd_to_datetime(night_events['sun_n12_rising'])],
                     2.5, 0.0, color='lightgray', alpha=0.3)
    ax.fill_between([mjd_to_datetime(night_events['sun_n12_rising']), 
                     mjd_to_datetime(night_events['sunrise'])],
                     2.5, 0.0, color='gray', alpha=0.3)
    ax.fill_between([mjd_to_datetime(night_events['sunrise'] - 2/24),
                    mjd_to_datetime(night_events['sun_n18_rising'])],
                    2.5, 0.0, color='pink', alpha=0.1)
    
    if not np.isnan(night_events['moonrise']):
        ax.axvline(mjd_to_datetime(night_events['moonrise']), linestyle='-', color='blue', alpha=0.3)
    if not np.isnan(night_events['moonset']):
        ax.axvline(mjd_to_datetime(night_events['moonset']), linestyle='-', color='red', alpha=0.3)
    
    colors = cc.glasbey_category10
    # Assign distinct target sets with different colors
    marker_colors = {}
    labels = {}
    count = 0
    if len(targets) > 0:
        for sp in targets.observation_reason.unique():
            marker_colors[sp] = colors[count]
            labels[sp] = sp
            count += 1
    
    if len(targets) > 0:
        visit_alpha = 0.7
        for sp in targets.observation_reason.unique():
            qq = targets.query("observation_reason == @sp")
            label = sp
            ax.plot(mjd_to_datetime(qq.mjd, 'tai'), qq.airmass, 
                    marker='o', linestyle='',
                    color=marker_colors[sp], label=label,
                    alpha=visit_alpha, markerfacecolor='none', zorder=3)
    
    ax.legend(loc=(1.01, 0.0), ncol=2)
    
    x0 = night_events['sunset']+30/60/24
    
    ax.set_xlim(mjd_to_datetime(night_events['sunset']+30/60/24), 
             mjd_to_datetime(night_events['sunrise']-30/60/24))
    ax_utc.set_xlim(mjd_to_datetime(night_events['sunset']+30/60/24, 'utc', timezone=tz_utc), 
             mjd_to_datetime(night_events['sunrise']-30/60/24, 'utc', timezone=tz_utc))
    
    ax.set_xlim(mjd_to_datetime(night_events['sunset']+30/60/24), 
             mjd_to_datetime(night_events['sunrise']-30/60/24))
    ax_utc.set_xlim(mjd_to_datetime(night_events['sunset']+30/60/24, 'utc', timezone=tz_utc), 
             mjd_to_datetime(night_events['sunrise']-30/60/24, 'utc', timezone=tz_utc))
    
    # Set ticks relevant sides
    ax.tick_params(axis="x", bottom=True, top=False, labelbottom=True, labeltop=False)
    ax_utc.tick_params(axis="x", bottom=False, top=True, labelbottom=False, labeltop=True)
    
    # Rotate and align bottom ticklabels
    plt.setp([tick.label1 for tick in ax.xaxis.get_major_ticks()], rotation=45,
             ha="right", va="center", rotation_mode="anchor")
    
    # Rotate and align top ticklabels
    plt.setp([tick.label2 for tick in ax_utc.xaxis.get_major_ticks()], rotation=45,
             ha="left", va="center",rotation_mode="anchor")
    
    plt.grid(True, alpha=0.2)
    
    plt.ylim(2.5, 0.9)
    
    ax.set_ylabel("Airmass", fontsize="large")
    ax.set_xlabel(f"Time ({tz})", fontsize="large")
    ax_utc.set_xlabel("Time (UTC)", fontsize='large')
    _ = plt.ylabel("Airmass", fontsize="large")

In [ ]:
#Time(pd.DataFrame(observations).query("observation_reason.str.contains('DDF')")['mjd'].max(), format='mjd').iso
#np.ptp(pd.DataFrame(observations).query("observation_reason.str.contains('DDF')")['mjd'].values) * 24

In [ ]:
make_plot_obs = True
if make_plot_obs:
    m_nvis = maf.CountMetric(col='mjd', metric_name = "Nvisits")
    s = maf.HealpixSlicer(nside=64, lon_col='RA', lat_col='dec', rot_sky_pos_col_name = 'rotSkyPos', lat_lon_deg=False)
    constraint = ''
    opsvis = pd.DataFrame(observations).query("observation_reason.str.contains('pair')").to_records()
    onvis = maf.MetricBundle(m_nvis, s, constraint)
    g = maf.MetricBundleGroup({f'nvisits': onvis}, None)
    g.run_current(constraint, opsvis)
    
    fig = make_sv_plot(onvis, proj='McBryde', vmin=0, vmax=3, add_bg=False)
    fig

In [ ]:
# sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(20250828, sun_alt=-12)
# print(sunset.iso, sunrise.iso)
# match = np.where((observatory.downtimes['end'] > sunset.mjd) & (observatory.downtimes['start'] < sunrise.mjd))
# print(observatory.downtimes[match])
# vv = visits.query("day_obs == 20250828")
# vv.obs_start_mjd.min(), vv.obs_start_mjd.max()

In [ ]:
# vv = visits.query("day_obs == @day_obs").copy()
#oo = pd.DataFrame(observations)
#print(len(vv), len(oo))
#oo.query("scheduler_note.str.contains('dd')")[['mjd', 'RA', 'dec', 'band', 'scheduler_note']]

In [ ]:
# compare observations to visits on this dayobs
# colors = rn_plots.PlotStyles.band_colors
# vv['hpid'] = hp.ang2pix(nside, vv.s_ra, vv.s_dec, nest=True, lonlat=True)
# oo['hpid'] = hp.ang2pix(nside, np.degrees(oo.RA), np.degrees(oo.dec), nest=True, lonlat=True)
# bands = set(vv.band.unique()).union(set(oo.band.unique()))
# for b in bands:
#     vvv = vv.query("band == @b")
#     plt.plot(vvv.obs_start_mjd, vvv.hpid, color=colors[b], 
#              marker='.', linestyle='', markersize=9, label=f"onsky {b}")
#     ooo = oo.query("band == @b")
#     plt.plot(ooo.mjd, ooo.hpid, markeredgecolor=colors[b], 
#              markerfacecolor='none', marker='o', linestyle='', markersize=10, label=f"sim {b}")
# plt.legend(loc=(1.01, 0.1))
# plt.xlabel("MJD")
# plt.ylabel("nested healpix id")

In [ ]:
observations_one = copy.deepcopy(observations)

In [ ]:
#vv = pd.DataFrame(observations).query("mjd > @sunset.mjd and mjd < @sunrise.mjd")
#vv.query("observation_reason.str.contains('DD')")[['mjd', 'band', 'alt', 'az', 'RA', 'dec', 'rotSkyPos', 'observation_reason', 'scheduler_note']]

In [ ]:
# start = observations['mjd'].min()
# vq = pd.DataFrame(scheduler.survey_lists[0][0].obs_wanted).query("flush_by_mjd > @start and mjd < @sunrise.mjd and observed == False")
# display(HTML(vq[['ID', 'RA', 'dec', 'mjd', 'flush_by_mjd', 'band', 'scheduler_note', 'observation_reason', 'moon_min_distance', 'alt_min']].to_html()))

In [ ]:
# qq = pd.DataFrame(scheduler.survey_lists[0][0].obs_wanted)

# ddf_visits = visits.query("observation_reason.str.contains('ddf')")
# for field in ddf_visits.observation_reason.unique():
#     q = ddf_visits.query("observation_reason == @field")
#     _ = plt.hist(q.HA, bins=np.arange(0, 24, 0.2), alpha=0.3, label=field)
# plt.legend()

# now = Time.now().mjd
# qqq = qq.query("mjd < @now")
# len(qqq.query("observed == True")) / len(qqq)

In [ ]:
if single_night:
    out_dir = f"single_{run_name}"
    print(out_dir, run_name)
!mkdir $out_dir
filename = os.path.join(out_dir, run_name + '.db')
print(filename)
!rm $filename

In [ ]:
bad_visit_ids = augment_visits.fetch_excluded_visits("lsstcam")
gvisits = augment_visits.exclude_visits(survey_info['consdb_visits'], bad_visit_ids)
g_initial_opsim = rn_sim.consdb_to_opsim(gvisits)
g_initial_opsim["note"] = g_initial_opsim["scheduler_note"].copy()

# sim_visits = svs.save_opsim(observatory=observatory, observations=[], 
#                             initial_opsim=g_initial_opsim, 
#                             filename=os.path.join(out_dir, run_name + '.db'))

sim_visits = g_initial_opsim 

In [ ]:
sim_visits['day_obs'] = np.array([rn_dayobs.day_obs_str_to_int(t.split("T")[0]) for t in 
                                  (Time(np.floor(sim_visits.observationStartMJD - 0.5) + 0.5, format='mjd', scale='tai').isot)])
sim_visits.day_obs.max()

In [ ]:
ddf_visits = sim_visits.query("observation_reason.str.contains('DD') or observation_reason.str.contains('ddf')").copy()
ddf_visits.loc[:, 'observation_reason'] = ddf_visits.observation_reason.str.lower()
ss = ddf_visits.groupby(["observation_reason", "band"]).agg({'observationStartMJD': 'count'})
ss = ss.reset_index('band').pivot(columns=["band"]).droplevel(0, axis=1)
ss['all'] = ss.sum(axis=1)
display(ss.query("observation_reason.str.contains('dd')")[['u', 'g', 'r', 'i', 'z', 'y', 'all']])

ss = ddf_visits.groupby(["observation_reason", "band"]).agg({'day_obs': 'nunique'})
ss = ss.reset_index('band').pivot(columns=["band"]).droplevel(0, axis=1)
ss['all'] = ddf_visits.groupby("observation_reason").agg({'day_obs': 'nunique'})
display(ss.query("observation_reason.str.contains('dd')")[['u', 'g', 'r', 'i', 'z', 'y', 'all']])

In [ ]:
# Did we manage to schedule the available time? 
observations = ObservationArray()
survey_info = svs.count_obstime(observations, survey_info)

plt.figure(figsize=(10,6))
plt.plot(survey_info['dayobsmjd'], survey_info['hours_in_night'], 'k')
w = 1
plt.bar(survey_info['dayobsmjd'], survey_info['downtime_per_night'], width=w, color='r')
plt.bar(survey_info['dayobsmjd'], survey_info['hours_in_night'] - survey_info['downtime_per_night'], bottom=survey_info['downtime_per_night'], width=w,
        color='lightblue', label="SV time per night")
plt.bar(survey_info['dayobsmjd'], survey_info['obs_time_per_night'],  bottom=survey_info['downtime_per_night'], 
        color='k', width=w, alpha=0.3, label='Observing')

plt.axvline(np.floor(Time.now().mjd - 0.5), color='b', linestyle=':')
plt.legend()
plt.ylabel("Hours per night", fontsize='large')
plt.title(f"SV surveys from {survey_info['survey_start'].iso[0:10]} to {survey_info['survey_end'].iso[0:10]}")

In [ ]:
visits.query("observation_reason.str.contains('too')").groupby("day_obs").agg({"obs_start_mjd": np.ptp}).sum() * 24

In [ ]:
#observatory.mjd = observatory.mjd - 0.4
# conditions = observatory.return_conditions()
# print(Time(conditions.mjd, format='mjd', scale='utc').iso)
# rewards = schedview_compute.make_scheduler_summary_df(scheduler, conditions)
# rewards

In [ ]:
# s = scheduler.survey_lists[5][1]
# # show why a survey might be masked out 
# footprintbf = [bf for bf in s.basis_functions if 'Footprint' in bf.label()]
# maskbf = [bf for bf in s.basis_functions if 'Avoid' in bf.label() or 'Shadow' in bf.label() or "HaMask" in bf.label()]
# mask = np.zeros(hp.nside2npix(32))
# for mbf in maskbf:
#     mask += mbf(conditions)
# mask = np.where(np.isnan(mask), 0.3, 1)
# fp = np.zeros(hp.nside2npix(32))
# for fbf in footprintbf:
#     fp += fbf(conditions)
# hp.mollview(fp, alpha=mask, rot=(200, 0, 0))
# hp.graticule()

In [ ]:
from rubin_scheduler.scheduler.utils import get_current_footprint
nside = 64
fp, labels = get_current_footprint(nside=nside)
tsurvey_info = svs.survey_times(verbose=True, no_downtime=True)
tsurvey_info.update(svc.survey_footprint(survey_start_mjd=tsurvey_info["survey_start"].mjd, nside=nside))
pp = tsurvey_info["skymap"]["map"]
alpha = np.where(pp >= 0, 1, fp['r'])
alpha = np.where(alpha > 1, 1, alpha)
bg_fp = np.where(fp['r'] == 0, np.nan, fp['r'])
bg_fp = np.where(bg_fp > 1, 1, bg_fp)
sv_fp = np.where(tsurvey_info['fp_array']['r'] > 0, 1, np.nan)

def make_sv_plot(metric_bundle, proj='laea', vmin=None, vmax=None, ax=None, add_bg=True, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    else:
        fig = ax.get_figure()
    
    if proj == 'laea':
        sp = skyproj.LaeaSkyproj( 
            ax=ax,
            celestial=True,
            galactic=False,
            gridlines=True,
            n_grid_lon=8,
            n_grid_lat=5,
            lat_0=-90,
            lon_0=0,
            extent=[0.0, 360.0, -90, 85],
        )
        # Laea only shows half the sky if zoom = True
        # due to bug in current skyproj
        zoom = False
    else: 
        sp = skyproj.McBrydeSkyproj( 
            ax=ax,
            celestial=True,
            galactic=False,
            gridlines=True,
            n_grid_lon=8,
            n_grid_lat=7,
            lon_0=0,
        )
        zoom = True

    if fig is not None and proj == 'laea':
        sp.ax.set_xlabel("R.A.", fontsize=12, labelpad=9)
        sp.ax.set_ylabel("Dec.", fontsize=12, labelpad=12)

    if add_bg:
        mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
            bg_fp, cmap='Greys', vmin=-1, vmax=4, nest=False, zoom=False, zorder=0
        )
        
        mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
            sv_fp, cmap='Blues', vmin=0, vmax=3, nest=False, zoom=False, zorder=1
        )
        zoom = False
    
    mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
        metric_bundle.metric_values.filled(np.nan), vmin=vmin, vmax=vmax, nest=False, zoom=zoom, zorder=1.5
    )

    if vmin is None and vmax is None:
        extend = None
    elif vmin is None and vmax is not None:
        extend = "max"
    elif vmin is not None and vmax is None:
        extend = "min"
    else:
        extend = "both"
    _ = sp.draw_colorbar(label=f"{metric_bundle.metric.name} {metric_bundle.info_label}", pad=0.12, shrink=0.5, extend=extend, location="bottom", orientation='horizontal')

    sp.ax.set_xlabel("R.A.", fontsize=12, labelpad=5)
    sp.ax.set_ylabel("Dec.", fontsize=12, labelpad=10)
    
    if title is not None:
        plt.title(title, pad=30)
    return fig

In [ ]:
plt.figure(figsize=(8, 6))


nights = np.arange(0, sim_visits.night.max())
_ = plt.hist(sim_visits.night, bins=nights, histtype='step', cumulative=True, label='sv_sim', linewidth=2)

survey_info['consdb_visits']['night'] = np.floor(survey_info['consdb_visits'].obs_start_mjd - 0.5) - Time("2025-06-20T12:00:00", scale='tai').mjd + 1

night_cut = rn_dayobs.day_obs_to_time(day_obs).mjd - Time("2025-06-20T12:00:00", scale='tai').mjd - 0.5

nights = np.arange(0, night_cut+1) # survey_info['consdb_visits'].night.max()+1)
#_ = plt.hist(survey_info['consdb_visits'].night, bins=nights, histtype='step', cumulative=True, label='consdb', linewidth=2)
    
#plt.axvline(survey_info['consdb_visits'].night.max() + 0.5, color='black', linewidth=2)
plt.axvline(night_cut+0.5, color='black', linewidth=2)

plt.xlim(0, sim_visits.night.max() - 1)
plt.grid(alpha=0.5)
plt.xlabel("Night", fontsize='x-large')
plt.ylabel("Cumulative number of visits", fontsize='x-large')
#plt.legend()

In [ ]:
sim_nvisits = {}
sim_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
    constraint = f"{b}"
    if b == 'all':
        opsvis = sim_visits.to_records()
    else:
        opsvis = sim_visits.query("band == @b").to_records()
    sim_nvisits[b] = maf.MetricBundle(m_nvis, s, constraint)
    sim_coadd[b] = maf.MetricBundle(m_coadd, s, constraint)
    g = maf.MetricBundleGroup({f'nvisits {b}': sim_nvisits[b], f'coadd {b}': sim_coadd[b]}, None)
    g.run_current(constraint, opsvis)

In [ ]:
for b in sim_nvisits:
    v = np.nanmedian(sim_nvisits[b].metric_values.filled(0) * sv_fp)
    d = np.nanmedian(sim_coadd[b].metric_values.filled(0) * sv_fp)
    print(b, v, f"{d:.2f}")

vmax = np.percentile(sim_nvisits['all'].metric_values.compressed(), 95)
fig = make_sv_plot(sim_nvisits['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title="SV sim")
fig.savefig("sv_sim_nvisits.png", bbox_inches='tight')

In [ ]:
consdbvisits = survey_info['consdb_visits']
too_visits = consdbvisits.query("target_name.str.contains('ToO')")
print('too onsky fraction', too_visits.exp_time.sum() / 60 / 60 / (consdbvisits.exp_time.sum()/60/60) * 100)

night_onsky = np.where(survey_info['dayobsmjd'] <= np.floor(Time.now().mjd - 0.5))
obs_hours = (survey_info['hours_in_night'][night_onsky] - survey_info['avail_per_night'][night_onsky]).sum()
too_hours = (too_visits.groupby('day_obs').agg({'obs_start_mjd': np.ptp})*24).values.sum() 
print("too onsky hours fraction", too_hours / obs_hours * 100)

In [ ]:
sim_nvisits_late = {}
sim_coadd_late = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
    constraint = f"{b}"
    if b == 'all':
        opsvis = sim_visits.query("observationId < 2025062000200").to_records()
    else:
        opsvis = sim_visits.query("observationId < 2025062000200 and band == @b").to_records()
    sim_nvisits_late[b] = maf.MetricBundle(m_nvis, s, constraint)
    sim_coadd_late[b] = maf.MetricBundle(m_coadd, s, constraint)
    g = maf.MetricBundleGroup({f'nvisits {b}': sim_nvisits_late[b], f'coadd {b}': sim_coadd_late[b]}, None)
    if len(opsvis) == 0:
        sim_nvisits_late[b].metric_values = np.ma.MaskedArray(np.zeros(len(s)), np.ones(len(s)))
        sim_coadd_late[b].metric_values = np.ma.MaskedArray(np.zeros(len(s)), np.ones(len(s)))
    else:
        g.run_current(constraint, opsvis)

In [ ]:
for b in sim_nvisits_late:
    v = np.nanmedian(sim_nvisits_late[b].metric_values.filled(0) * sv_fp)
    d = np.nanmedian(sim_coadd_late[b].metric_values.filled(0) * sv_fp)
    print(b, v, f"{d:.2f}")

vmax = np.percentile(sim_nvisits_late['all'].metric_values.compressed(), 95)
fig = make_sv_plot(sim_nvisits_late['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title="SV sim post-update")
fig.savefig("sv_sim_nvisits_late.png", bbox_inches='tight')

In [ ]:
print("% of the way through SV period", (Time.now() - survey_info['survey_start']) / (survey_info['survey_end'] - survey_info['survey_start']))
print("% of the number of predicted visits", len(survey_info['consdb_visits']) / len(sim_visits))
bad_visit_ids = augment_visits.fetch_excluded_visits("lsstcam")
gvisits = augment_visits.exclude_visits(survey_info['consdb_visits'], bad_visit_ids)
print("# good visits, % of the number of predicted visits", len(gvisits), len(gvisits) / len(sim_visits))
print("good visits post 20250719", len(gvisits.query("day_obs > 20250719")), len(gvisits), len(gvisits.query("day_obs > 20250724"))/len(gvisits),)
print("# of days in last period", (np.floor(Time.now().mjd - 0.5)+0.5 - Time("2025-07-19T12:00:00").mjd))


In [ ]:
sim_nvisits_now = {}
sim_coadd_now = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
    constraint = f"{b}"
    if b == 'all':
        opsvis = sim_visits.query("observationId > 2025062000200 ").to_records()
    else:
        opsvis = sim_visits.query("observationId > 2025062000200 and band == @b").to_records()
    sim_nvisits_now[b] = maf.MetricBundle(m_nvis, s, constraint)
    sim_coadd_now[b] = maf.MetricBundle(m_coadd, s, constraint)
    g = maf.MetricBundleGroup({f'nvisits {b}': sim_nvisits_now[b], f'coadd {b}': sim_coadd_now[b]}, None)
    g.run_current(constraint, opsvis)

In [ ]:
for b in sim_nvisits_now:
    v = np.nanmedian(sim_nvisits_now[b].metric_values.filled(0) * sv_fp)
    d = np.nanmedian(sim_coadd_now[b].metric_values.filled(0) * sv_fp)
    print(b, v, f"{d:.2f}")

vmax = np.percentile(sim_nvisits_now['all'].metric_values.compressed(), 95)
fig = make_sv_plot(sim_nvisits_now['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title="SV sim 2.0 onsky to date")
fig.savefig("sv_sim_2.0_nvisits_now.png", bbox_inches='tight')

In [ ]:
np.where(np.isfinite(sv_fp))[0].size *hp.nside2pixarea(64, degrees=True)

In [ ]:
# bins = np.arange(0.5, 2.6, 0.001)
# for b in 'ugrizy':
#     q = survey_info['consdb_visits'].query("band == @b")
#     _ = plt.hist(q.fwhm_eff, bins=bins, cumulative=True, density=True, color=band_colors[b], histtype='step', linestyle='-.', label=f"SV Consdb {b}")
#     print('sv consdb', b, f"{q.fwhm_eff.median():.2f}")
#     q = sim_visits.query("band == @b and day_obs < 20250801")
#     _ = plt.hist(q.seeingFwhmEff, bins=bins, cumulative=True, density=True, color=band_colors[b], histtype='step', linestyle='-', label=f"SV Sim {b}")
#     print('sim', b, f"{q.seeingFwhmEff.median():.2f}")

# plt.axhline(.5, color='gray', alpha=0.7)
# plt.legend(loc=(1.01, 0.1))
# plt.grid(alpha=0.4)
# plt.xlim(0.4, 2.0)
# plt.xlabel("FWHM (arcsec)")

In [ ]:
# bad_visits = augment_visits.fetch_excluded_visits("lsstcam")
# gv = augment_visits.exclude_visits(consdbvisits, bad_visits)
# len(gv)

In [ ]:
#gv, slews = rn_sch.add_model_slew_times(gv, endpoints['efd'], model_settle=3.45)